# Train — TCR–pMHC pose-binding models

**Training-only driver.** All model architectures, geometry, and training utilities
live in `tcrpmhc_pose_binding/src/` and are imported here — this notebook only loads
data, pretrains, runs the per-peptide and across-peptide evaluations, and plots.

**Setup:** put the repo on Drive at `REPO` (so `REPO/src/*.py` exist) and the three
CSVs in `REPO/data/`. Runtime → GPU.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys, os
REPO    = "/content/drive/MyDrive/tcrpmhc_pose_binding"   # contains src/ and data/
DATA_DIR= f"{REPO}/data"
OUT_DIR = f"{REPO}/results"; os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, f"{REPO}/src")
print("src on path:", os.path.isdir(f"{REPO}/src"), "| data:", os.path.isdir(DATA_DIR))

In [ ]:
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from data import load_data
from train_utils import (pretrain_tcr_encoders, make_warm_start, pretrain_posevae,
                         run_repeated, aggregate_oof, cluster_bootstrap, DEVICE)
from peptide_specific import run_per_peptide
from across_peptide import PeptideAwareFusion, CondPeptideAware, CondPeptideAwareFusion, run_mixed, run_lopo
print("device:", DEVICE)

## 1. Load data (ipTM ≥ 0.5) + pretrain (Stage 0 TCR encoders, Stage 1 pose VAEs)

In [ ]:
D = load_data(DATA_DIR, min_iptm=0.5)
pool, seq, trans_scale, POSE_ALL = D["pool"], D["seq"], D["trans_scale"], D["POSE_ALL"]
print("pool:", len(pool), "| peptides:", sorted(pool.peptide.unique()))
print(pool.groupby("peptide").label.agg(["sum","count"]))

warm_start = make_warm_start(pretrain_tcr_encoders(seq))      # Stage 0 (all TCRs)
vae6 = pretrain_posevae(POSE_ALL, "6d")                       # Stage 1 (all structures)
vaeq = pretrain_posevae(POSE_ALL, "qnn")
print("pretraining done")

## 2. Peptide-specific (fixed antigen, per peptide)

In [ ]:
resA = run_per_peptide(pool, trans_scale, vae6, vaeq, warm_start)
resA.to_csv(f"{OUT_DIR}/partA_perpeptide.csv", index=False)
print(resA.pivot_table(index="peptide", columns="model", values="auroc").round(3))
resA

## 3. Across-peptide (peptide tower) — MIXED + LOPO

In [ ]:
AFS = {
 "seq+pep":                 lambda: warm_start(PeptideAwareFusion(pose_vae=None)),
 "seq+pep+pose_6d":         lambda: warm_start(PeptideAwareFusion(pose_vae=vae6)),
 "seq+pep+pose_qnn":        lambda: warm_start(PeptideAwareFusion(pose_vae=vaeq)),
 "cond_pose_6d":            lambda: warm_start(CondPeptideAware(rotation_encoder="6d",  cond_on="vpep")),
 "cond_pose_qnn":           lambda: warm_start(CondPeptideAware(rotation_encoder="qnn", cond_on="vpep")),
 "seq+pep+cond_pose_6d":    lambda: warm_start(CondPeptideAwareFusion(rotation_encoder="6d",  cond_on="vpep")),
 "seq+pep+cond_pose_qnn":   lambda: warm_start(CondPeptideAwareFusion(rotation_encoder="qnn", cond_on="vpep")),
}
AUXSET = {k for k in AFS if "cond_pose" in k}   # all conditional arms get recon+KL (fair)
mixed_df, oofs, yM, groups = run_mixed(pool, trans_scale, AFS, AUXSET=AUXSET, use_mismatch=False)
lopo_df = run_lopo(pool, trans_scale, AFS, AUXSET=AUXSET, use_mismatch=False)
resB = pd.concat([mixed_df, lopo_df], ignore_index=True); resB.to_csv(f"{OUT_DIR}/partB_across.csv", index=False)
print("MIXED mean+/-std:")
print(mixed_df.groupby("model")[["auroc","auprc"]].agg(["mean","std"]).round(3))

### Cluster-bootstrap contrasts (95% CI, per-sample aggregated, MIXED)

In [ ]:
for cn,(a,b) in {"(seq+pep+pose_6d)-(seq+pep)":("seq+pep+pose_6d","seq+pep"),
                 "(seq+pep+cond_pose_6d)-(seq+pep)":("seq+pep+cond_pose_6d","seq+pep"),
                 "(seq+pep+cond_pose_6d)-(seq+pep+pose_6d)":("seq+pep+cond_pose_6d","seq+pep+pose_6d"),
                 "(cond_pose_6d)-(cond_pose_qnn)":("cond_pose_6d","cond_pose_qnn")}.items():
    ta = oofs[a][1] & oofs[b][1]
    (dR,loR,hiR,pR),(dP,loP,hiP,pP) = cluster_bootstrap(yM, oofs[a][0], oofs[b][0], groups, ta)
    print(f"[{cn}] dAUROC={dR:+.3f} CI[{loR:+.3f},{hiR:+.3f}]  dAUPRC={dP:+.3f} CI[{loP:+.3f},{hiP:+.3f}]")

## 4. Figures

In [ ]:
# Part A
cols=[m for m in ["seq","pose_only_6d","pose_only_qnn","seq+pose_6d","seq+pose_qnn",
                  "FramePose_RF","FramePose_ExtraTrees","FramePose_HistGBT"] if m in set(resA.model)]
pa=resA.pivot_table(index="peptide",columns="model",values="auroc")[cols]
sa=resA.pivot_table(index="peptide",columns="model",values="auroc_sd")[cols]
ax=pa.plot.bar(yerr=sa,capsize=2,figsize=(15,5)); ax.set_ylim(0,1); ax.axhline(0.5,ls="--",c="gray")
ax.set_title("Part A AUROC (per peptide, 10x repeated 80/20 grouped CV; err=std)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/partA.png",dpi=150); plt.show()
# Part B
orderB=[m for m in AFS if m in set(resB.model)]
sB=resB.groupby(["regime","model"])[["auroc","auprc"]].agg(["mean","std"])
fig,axB=plt.subplots(1,2,figsize=(16,5)); reg=[r for r in ["LOPO","MIXED"] if r in set(resB.regime)]
x=np.arange(len(reg)); w=0.8/max(len(orderB),1)
for j,met in enumerate(["auroc","auprc"]):
    for i,m in enumerate(orderB):
        mean=[sB.loc[(r,m),(met,"mean")] if (r,m) in sB.index else np.nan for r in reg]
        sd=[sB.loc[(r,m),(met,"std")] if (r,m) in sB.index else 0 for r in reg]
        axB[j].bar(x+(i-(len(orderB)-1)/2)*w,mean,w,yerr=sd,capsize=3,label=m)
    axB[j].set_xticks(x); axB[j].set_xticklabels(reg); axB[j].set_ylim(0,1); axB[j].axhline(0.5,ls="--",c="gray")
    axB[j].set_title("Part B "+met.upper()); axB[j].legend(fontsize=6)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/partB.png",dpi=150); plt.show()
print("saved to", OUT_DIR)